In [2]:
import pandas as pd

# Load data
train = pd.read_csv("../data/raw/train.tsv", sep="\t")
test = pd.read_csv("../data/raw/test.tsv", sep="\t")

# Basic info
print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nColumns:", train.columns)

# Sample rows
print("\nSample data:")
print(train.head())

# Class distribution
print("\nSentiment distribution:")
print(train["Sentiment"].value_counts())

# Check text length
train["length"] = train["Phrase"].apply(lambda x: len(x.split()))

print("\nText length stats:")
print(train["length"].describe())

# Longest / shortest examples
print("\nShortest phrase:")
print(train.loc[train["length"].idxmin()]["Phrase"])

print("\nLongest phrase:")
print(train.loc[train["length"].idxmax()]["Phrase"])

Train shape: (156060, 4)
Test shape: (66292, 3)

Columns: Index(['PhraseId', 'SentenceId', 'Phrase', 'Sentiment'], dtype='str')

Sample data:
   PhraseId  SentenceId                                             Phrase  \
0         1           1  A series of escapades demonstrating the adage ...   
1         2           1  A series of escapades demonstrating the adage ...   
2         3           1                                           A series   
3         4           1                                                  A   
4         5           1                                             series   

   Sentiment  
0          1  
1          2  
2          2  
3          2  
4          2  

Sentiment distribution:
Sentiment
2    79582
3    32927
1    27273
4     9206
0     7072
Name: count, dtype: int64

Text length stats:
count    156060.000000
mean          7.203364
std           7.024604
min           0.000000
25%           2.000000
50%           5.000000
75%          10.000000
ma

In [3]:
import pandas as pd
import torch
from collections import Counter


In [15]:
# Config
MAX_LEN = 25
MIN_FREQ = 2

train = pd.read_csv("../data/raw/train.tsv", sep="\t")
test = pd.read_csv("../data/raw/test.tsv", sep="\t")

train["Phrase"] = train["Phrase"].fillna("")
test["Phrase"] = test["Phrase"].fillna("")

print(train.shape, test.shape)
train.head()

(156060, 4) (66292, 3)


,PhraseId,SentenceId,Phrase,Sentiment
0,1,1,A series of escapades demonstrating the adage ...,1
1,2,1,A series of escapades demonstrating the adage ...,2
2,3,1,A series,2
3,4,1,A,2
4,5,1,series,2


In [16]:
def tokenize(text):
    return text.lower().split()

In [17]:
def build_vocab(texts, min_freq=2):
    counter = Counter()

    for text in texts:
        counter.update(tokenize(text))

    vocab = {"<PAD>": 0, "<UNK>": 1}

    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)

    return vocab

vocab = build_vocab(train["Phrase"], MIN_FREQ)

print("Vocab size:", len(vocab))


import json

# convert keys to string-safe format (already strings usually)
with open("../data/vocab.json", "w") as f:
    json.dump(vocab, f)

Vocab size: 16507


In [18]:
def encode(text, vocab):
    tokens = tokenize(text)
    ids = [vocab.get(word, vocab["<UNK>"]) for word in tokens]

    # truncate
    ids = ids[:MAX_LEN]

    # pad
    if len(ids) < MAX_LEN:
        ids += [vocab["<PAD>"]] * (MAX_LEN - len(ids))

    return torch.tensor(ids)

In [19]:
def prepare_dataset(df, vocab):
    X = torch.stack([encode(text, vocab) for text in df["Phrase"]])

    if "Sentiment" in df.columns:
        y = torch.tensor(df["Sentiment"].values)
        return X, y

    return X

X_train, y_train = prepare_dataset(train, vocab)

print("X shape:", X_train.shape)
print("y shape:", y_train.shape)

X shape: torch.Size([156060, 25])
y shape: torch.Size([156060])


In [20]:
X_test = prepare_dataset(test, vocab)

print("Test shape:", X_test.shape)

Test shape: torch.Size([66292, 25])


In [22]:
torch.save(X_train, "../data/X_train.pt")
torch.save(y_train, "../data/y_train.pt")
torch.save(X_test, "../data/X_test.pt")